# Octodon Degus: Single Cell Analysis Template
Use this notebook to interactively inspect a single cell using the refactored pipeline.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure scripts directory is in path
sys.path.append(os.path.abspath('scripts'))

from utils.data_loader import load_cell_data
from utils.maps import firing_map, rate_map
from utils.modeling import get_gam_posicion, graficar_gam_posicion
from utils.cross_validation import generate_all_splits, find_optimal_lambda_dynamic

## 1. Load Data
Define the `Cell_ID` and let `load_cell_data` find the correct databases and epochs.

In [ ]:
cell_id = 'XII-OF-I_T1_N1'
data = load_cell_data(cell_id, bin_size_sec=0.1)

if data:
    x_bins, y_bins, ang_bins_rad, spikes = data
    print(f'Loaded {len(x_bins)} bins. Total spikes: {np.sum(spikes)}')
else:
    print('Failed to load data.')

## 2. Visualize Rate Maps
Using the `maps` module, we can visualize the raw tracking data.

In [ ]:
# Plot a simple scatter of the binned data:

plt.figure(figsize=(6,6))
plt.scatter(x_bins, y_bins, c='lightgray', s=1)
mask_spikes = spikes > 0
plt.scatter(x_bins[mask_spikes], y_bins[mask_spikes], c='red', s=5, alpha=0.5)
plt.title(f'Firing Map: {cell_id}')
plt.axis('equal')
plt.show()

## 3. GAM Modeling (Spatial)

In [ ]:
# We can train a GAM directly using the binned X, Y coords.
X_pos = np.column_stack((x_bins, y_bins))
Y = spikes

# Train with arbitrary lambda or use dynamic search
modelo_pos = get_gam_posicion(X_pos, Y, cell_id=cell_id, splines=9, lam=1.0)
graficar_gam_posicion(modelo_pos, X_pos, Y, title=f'GAM Pos: {cell_id}')